1. dividing the data to bins/sliding windows - 
2. calculating the linear model
    a) add weighting per window to match sample to population - using balance package, and post-stratification weighting (to match the known distribution based on the israeli Central Bureau of Statistic )
3. creating coef_df (for each linear model summary - coefficients etc.) - run if not using the coef_df created in step9

In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from tqdm import tqdm
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
from balance import Sample  # The core package logic

# --- CONFIGURATION ---
min_age = 20
max_age = 65

# Options: 'fixed', 'sliding_n_subjects', or 'sliding_n_years'
STRATEGY = 'fixed' 

# manually fixed:
# bins = [20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 100]


# Parameters for 'sliding_n_subjects'
window_subjects = 100 
step_subjects = 50

# Parameters for 'sliding_n_years'
window_years = 5
step_years = 1

# Parameters for 'fixed'
bin_size = 5

# Set to None if you want to run all ROIs
ROI_FILTER = [421, 422]

INFO (2026-03-11 18:20:38,614) [__init__/<module> (line 72)]: Using balance version 0.16.0



balance (Version 0.16.0) loaded:
    📖 Documentation: https://import-balance.org/
    🛠️ Help / Issues: https://github.com/facebookresearch/balance/issues/
    📄 Citation:
        Sarig, T., Galili, T., & Eilat, R. (2023).
        balance - a Python package for balancing biased data samples.
        https://arxiv.org/abs/2307.06024

    Tip: You can view this message anytime with balance.help()



# Data Prep

In [5]:
# --- 1. DATA LOADING & PREP ---
combined_df = pd.read_pickle('/home/gaia/Projects/legacy_data/best_combined_gm_volumes.pkl')
volumes = combined_df[(combined_df['classification_label'] == 1) | (combined_df['source'] == 'snbb')].copy()
volumes['age_in_years'] = pd.to_numeric(volumes['age_in_years'], errors='coerce')
volumes = volumes[(volumes['age_in_years'] >= min_age) & (volumes['age_in_years'] <= max_age)]
volumes = volumes.dropna(subset=['age_in_years', 'birth_year', 'sex', 'tiv'])


# windowing

In [ ]:
# --- 2. WINDOW DEFINITION ---
mappings = []

# Prepare sorted metadata for sliding strategies
meta = (
    volumes[['session_id', 'age_in_years']]
    .drop_duplicates('session_id')
    .sort_values('age_in_years')
    .reset_index(drop=True)
)

if STRATEGY == 'fixed':
    bins = np.arange(min_age, max_age + bin_size, bin_size)
    volumes['age_bin'] = pd.cut(volumes['age_in_years'], bins=bins)
    volumes = volumes.dropna(subset=['age_bin'])

elif STRATEGY == 'sliding_n_subjects':
    session_ids = meta['session_id'].values
    for i in range(0, len(session_ids) - window_subjects + 1, step_subjects):
        subset_ids = session_ids[i : i + window_subjects]
        age_min, age_max = meta.iloc[i]['age_in_years'], meta.iloc[i + window_subjects - 1]['age_in_years']
        label = f"({age_min:.1f}-{age_max:.1f})"
        for sid in subset_ids:
            mappings.append({'session_id': sid, 'age_bin': label})

elif STRATEGY == 'sliding_n_years':
    min_age = meta['age_in_years'].min()
    max_age = meta['age_in_years'].max()
    
    # Iterate through age ranges
    current_start = min_age
    window_idx = 0
    while (current_start + window_years) <= max_age:
        current_end = current_start + window_years
        
        # Find session IDs within this specific age range
        mask = (meta['age_in_years'] >= current_start) & (meta['age_in_years'] < current_end)
        subset_ids = meta.loc[mask, 'session_id'].values
        
        if len(subset_ids) > 0:
            label = f"({current_start:.1f}-{current_end:.1f})"
            for sid in subset_ids:
                mappings.append({'session_id': sid, 'age_bin': label})
        
        current_start += step_years
        window_idx += 1

# Merge back for all sliding strategies
if 'sliding' in STRATEGY:
    map_df = pd.DataFrame(mappings)
    volumes = volumes.merge(map_df, on='session_id')

# Analysis and weighting

In [7]:
# --- 3. ANALYSIS WITH BALANCE PACKAGE ---
atlas_csv = pd.read_csv("/home/gaia/Projects/legacy_data/my_master/space-MNI152_atlas-schaefer2018tian2020_res-1mm_den-400_div-7networks_dseg.csv")
results = []

if 'ROI_FILTER' in locals() and ROI_FILTER is not None:
    volumes = volumes[volumes['region_label'].isin(ROI_FILTER)]

for bin_label, df_bin in tqdm(volumes.groupby('age_bin', observed=True), desc="Processing Age Windows"):
    
    # 1. Get unique subjects for this window
    df_window_subjects = df_bin.drop_duplicates(subset=['subject_id'], keep='first').copy()
    if len(df_window_subjects) < 20:
        continue

    # 2. CREATE THE TARGET (Simulated Population)
    # We want a target where every birth year in this window has equal probability (Uniform)
    unique_years = df_window_subjects['birth_year'].unique()
    target_df = pd.DataFrame({'birth_year': unique_years})
    
    # 3. USE BALANCE TO ADJUST
    # 'sample' is your data, 'target' is the uniform distribution
    s = Sample.from_frame(df_window_subjects, outcome_columns=[])
    target = Sample.from_frame(target_df)
    
    # Adjust using 'null' method for simple frequency balancing, 
    # or 'post_stratify' to match the target years exactly
    adjusted_s = s.adjust(target, method="post_stratify")
    
    # 4. EXTRACT WEIGHTS & DIAGNOSTICS
    df_window_subjects['ps_weight'] = adjusted_s.get_weights()
    current_ess = adjusted_s.ess()
    
    # 5. BALANCE VISUALIZATIONS
    print(f"--- Diagnostics for Age Window: {bin_label} ---")
    # This replaces your manual histograms with professional plots
    # It shows the "Standardized Mean Difference" which is a great balance metric
    adjusted_s.plot_unadjusted_vs_adjusted() 
    plt.show()

    # 6. REGRESSION LOOP
    weight_lookup = df_window_subjects.set_index('subject_id')['ps_weight']
    
    for roi, df_roi in df_bin.groupby('region_label', observed=True):
        df_roi = df_roi.drop_duplicates(subset=['subject_id'], keep='first').copy()
        df_roi['ps_weight'] = df_roi['subject_id'].map(weight_lookup)
        
        try:
            model = smf.wls(
                'volume_mm3 ~ birth_year + C(sex) + tiv + age_in_years', 
                data=df_roi,
                weights=df_roi['ps_weight']
            ).fit()
            
            for var in model.params.index:
                results.append({
                    'age_bin': bin_label,
                    'region_label': roi,
                    'variable': var,
                    'coef': model.params[var],
                    't': model.tvalues[var],
                    'p': model.pvalues[var],
                    'n_subjects': len(df_roi),
                    'ess': current_ess
                })
        except:
            continue

Processing Age Windows:   0%|          | 0/9 [00:00<?, ?it/s]


ValueError: Error while inferring id_column from DataFrame. Specify a valid 'id_column' or provide 'id_column_candidates'. Original error: Cannot guess id column name for this DataFrame. None of the possible_id_columns candidates ['id'] were found in the DataFrame columns ['subject_id', 'session_id', 'region_label', 'tissue_type', 'volume_mm3', 'tiv', 'sex', 'institute', 'manufacturer', 'age_in_years', 'dob', 'gm_volume_cm3', 'protocol', 'source', 'birth_year', 'unnamed: 0', 'atlas_name', 'scan_time', 'age_at_scan', 'weight', 'directory_path', 'estimated_critical_info', 'scan_date', 'file_path', 'scan_year', 'classification_label', 'age_bin']. Please provide a value in column_name or update possible_id_columns to match an existing column.

# post process

In [ ]:
# --- 4. POST-PROCESSING ---
coef_df = pd.DataFrame(results)

# Add Atlas names
coef_df = coef_df.merge(atlas_csv[['index', 'name']], left_on='region_label', right_on='index', how='left')
coef_df.rename(columns={'name': 'region_name'}, inplace=True)

# Multiple Comparison Correction (FDR) per window for 'birth_year'
final_rows = []
for label, group in coef_df.groupby('age_bin'):
    mask = group['variable'] == 'birth_year'
    if mask.any():
        _, fdr_p, _, _ = multipletests(group.loc[mask, 'p'], method='fdr_bh')
        group.loc[mask, 'fdr_p'] = fdr_p
    final_rows.append(group)

coef_df = pd.concat(final_rows)

print("Analysis Complete.")

: 

: 

: 

: 

: 

# save

In [ ]:
# # --- 5. SAVE ---
# suffix = f"sliding_ws{window_subjects}_ss{step_subjects}" if STRATEGY == 'sliding_n_subjects' else f"sliding_wy{window_years}_sy{step_years}" if STRATEGY == 'sliding_n_years' else f"fixed_bin{bin_size}"



: 

: 

: 

: 

: 